In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = "/content/drive/MyDrive/factory-safety-ai-cctv"
LOCAL_PROJECT = "/content/factory-safety-ai-cctv"

!rm -rf "$LOCAL_PROJECT"
!cp -r "$DRIVE_PROJECT" "$LOCAL_PROJECT"

%cd "$LOCAL_PROJECT"
!ls

In [ ]:
# Colab already provides notebook/Jupyter runtime packages.
# Install only the packages needed for this notebook to avoid Colab dependency conflicts.
!pip install -q ultralytics nbformat pyyaml opencv-python pillow pandas matplotlib seaborn tqdm huggingface_hub

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ultralytics:", ultralytics.__version__)

In [ ]:
!rm -rf dataset/roboflow_hard_hat_workers_yolov11_raw
!mkdir -p dataset/roboflow_hard_hat_workers_yolov11_raw
!unzip -q dataset/roboflow_hard_hat_workers_yolov11_raw.zip -d dataset/roboflow_hard_hat_workers_yolov11_raw

!find dataset/roboflow_hard_hat_workers_yolov11_raw -maxdepth 4 -name "data.yaml"

# YOLOv11s PPE Candidate

YOLOv11s is the **primary candidate** for the PPE perception layer. This notebook mirrors the YOLOv8s baseline workflow so both models can be compared fairly using the same dataset YAML, sample images, clean FPS benchmark, qualitative visualizations, and risk-signal readiness preview.


## Environment Setup

This notebook should be run from the project root. It uses relative, Windows-friendly paths and allows Ultralytics to download model weights automatically.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dataset").exists() and (candidate / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Could not locate project root. Run this notebook from factory-safety-ai-cctv/ "
        "or open it from inside that project."
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

try:
    from ultralytics import YOLO
except ImportError as exc:
    YOLO = None
    print("ERROR: Ultralytics is required for model experiments.")
    print("Install with: pip install ultralytics")

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

## Dataset Config Check

For the 10/30-epoch comparison, this notebook must use the processed project dataset only: `data/processed/ppe_yolo/data.yaml` with schema `0=person, 1=helmet, 2=head`. It intentionally does **not** fall back to raw Roboflow YAML.

In [ ]:
import json
from pathlib import Path
import yaml

# IMPORTANT:
# For the 10/30-epoch comparison, do NOT fall back to the raw Roboflow YAML.
# We only accept the processed project schema:
#   0 = person, 1 = helmet, 2 = head

PROCESSED_DATASET_ROOT = Path("data/processed/ppe_yolo")
DATA_YAML = PROCESSED_DATASET_ROOT / "data.yaml"

IMAGE_EXTENSIONS_FOR_DATA_CHECK = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PROJECT_CLASS_SCHEMA = {
    0: "person",
    1: "helmet",
    2: "head",
}


def _read_simple_yaml_field(yaml_path: Path, key: str) -> str | None:
    if not yaml_path.exists():
        return None
    try:
        cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8")) or {}
        value = cfg.get(key)
        if isinstance(value, str):
            return value
    except Exception:
        pass

    for raw_line in yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        current_key, value = line.split(":", 1)
        if current_key.strip() == key:
            return value.strip().strip("'\"")
    return None


def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for item in path.rglob("*") if item.suffix.lower() in IMAGE_EXTENSIONS_FOR_DATA_CHECK)


def _count_labels(path: Path) -> int:
    if not path.exists():
        return 0
    return len(list(path.rglob("*.txt")))


def _normalize_names(names) -> dict[int, str]:
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    if isinstance(names, list):
        return {i: str(v) for i, v in enumerate(names)}
    raise AssertionError(f"Unsupported names format in data.yaml: {type(names)}")


def _resolve_dataset_root(yaml_path: Path, cfg: dict) -> Path:
    path_value = cfg.get("path")
    if not path_value:
        return yaml_path.parent.resolve()

    candidates = [
        Path(path_value),
        yaml_path.parent / path_value,
        PROJECT_ROOT / path_value,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    # Return the project-root candidate for clearer error messages.
    return (PROJECT_ROOT / path_value).resolve()


def _resolve_split_dir(yaml_path: Path, cfg: dict, split_key: str) -> Path:
    split_value = cfg.get(split_key)
    if not split_value:
        raise AssertionError(f"data.yaml missing split key: {split_key}")

    dataset_root = _resolve_dataset_root(yaml_path, cfg)
    candidate = dataset_root / split_value
    return candidate.resolve()


assert DATA_YAML.exists(), (
    f"Missing processed dataset YAML: {DATA_YAML}\n"
    "Bạn cần chạy notebook 05 và sync data/processed/ppe_yolo về Drive trước. "
    "Notebook này cố tình không fallback về Roboflow raw nữa để tránh train sai schema."
)

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f) or {}

names = _normalize_names(data_cfg.get("names"))
assert names == PROJECT_CLASS_SCHEMA, (
    f"Wrong class schema: {names}\n"
    f"Expected: {PROJECT_CLASS_SCHEMA}"
)

train_img_dir = _resolve_split_dir(DATA_YAML, data_cfg, "train")
val_img_dir = _resolve_split_dir(DATA_YAML, data_cfg, "val")
test_img_dir = _resolve_split_dir(DATA_YAML, data_cfg, "test")

dataset_root = _resolve_dataset_root(DATA_YAML, data_cfg)
train_lbl_dir = dataset_root / "labels/train"
val_lbl_dir = dataset_root / "labels/val"
test_lbl_dir = dataset_root / "labels/test"

dataset_counts = {
    "train_images": _count_images(train_img_dir),
    "val_images": _count_images(val_img_dir),
    "test_images": _count_images(test_img_dir),
    "train_labels": _count_labels(train_lbl_dir),
    "val_labels": _count_labels(val_lbl_dir),
    "test_labels": _count_labels(test_lbl_dir),
}

print(json.dumps(dataset_counts, indent=2))
print("Resolved dataset root:", dataset_root)
print("Resolved train dir:", train_img_dir)
print("Resolved val dir:", val_img_dir)
print("Resolved test dir:", test_img_dir)
print("Class schema:", names)

assert dataset_counts["train_images"] > 0, "Processed train split is empty."
assert dataset_counts["val_images"] > 0, "Processed val split is empty. Run the val split cell in notebook 05."
assert dataset_counts["test_images"] > 0, "Processed test split is empty."
assert dataset_counts["train_labels"] > 0, "Processed train labels are empty."
assert dataset_counts["val_labels"] > 0, "Processed val labels are empty."
assert dataset_counts["test_labels"] > 0, "Processed test labels are empty."

print("\n✅ Using processed project dataset only:")
print(DATA_YAML)


## Model Load

Load `yolo11s.pt` with Ultralytics YOLO.

In [ ]:
MODEL_ID = "yolo11s_ppe"
BASE_MODEL = "yolo11s.pt"
ROLE = "primary_candidate"
EXPERIMENT_DIR = Path("experiments/yolo11s_ppe")
RUNS_DIR = EXPERIMENT_DIR / "runs"
METRICS_PATH = EXPERIMENT_DIR / "metrics.json"

EPOCHS = 10
IMGSZ = 640
BATCH = 8
model = None
model_load_error = None
train_result = None
val_metrics = None
train_error = None
val_error = None
if YOLO is None:
    model_load_error = "Ultralytics is not installed."
    print("Skipping model load because Ultralytics is unavailable.")
else:
    try:
        model = YOLO(BASE_MODEL)
        print("Model loaded:", BASE_MODEL)
        print("Model names:", getattr(model, "names", "N/A"))
    except Exception as exc:
        model_load_error = str(exc)
        print(f"Model load failed gracefully: {exc}")
print(f"Training config: EPOCHS={EPOCHS}, IMGSZ={IMGSZ}, BATCH={BATCH}")


## Training on Processed Dataset

This run trains on the processed project dataset. Use `EPOCHS = 10` for preliminary comparison, then `EPOCHS = 30` only after the processed 10-epoch pipeline is confirmed.

In [ ]:
train_result = None
if DATA_YAML is None:
    print("Skipping training because DATA_YAML is missing.")
elif model is None:
    print("Skipping training because model is not loaded.")
else:
    try:
        TRAIN_PROJECT = RUNS_DIR.resolve()
        TRAIN_PROJECT.mkdir(parents=True, exist_ok=True)
        print("Training project:", TRAIN_PROJECT)
        train_result = model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            project=str(TRAIN_PROJECT),
            name="smoke_train",
            exist_ok=True,
            device=DEVICE,
        )
    except Exception as exc:
        train_error = str(exc)
        print(f"Training failed gracefully: {exc}")


## Validation

Run validation after smoke training if `DATA_YAML` is available.

In [ ]:
val_metrics = None
if DATA_YAML is None:
    print("Skipping validation because DATA_YAML is missing.")
elif model is None:
    print("Skipping validation because model is not loaded.")
else:
    try:
        VAL_PROJECT = RUNS_DIR.resolve()
        VAL_PROJECT.mkdir(parents=True, exist_ok=True)
        print("Validation project:", VAL_PROJECT)
        val_metrics = model.val(
            data=str(DATA_YAML),
            project=str(VAL_PROJECT),
            name="val",
            exist_ok=True,
            device=DEVICE,
        )
        print("Validation completed.")
    except Exception as exc:
        val_error = str(exc)
        print(f"Validation failed gracefully: {exc}")
        val_metrics = None


## Save Metrics

Extract available validation metrics safely. Unavailable values stay `TBD`.

In [ ]:
def metric_value(obj, attr_path: str, default="TBD"):
    current = obj
    for attr in attr_path.split("."):
        if current is None:
            return default
        current = getattr(current, attr, None)
    if current is None:
        return default
    try:
        if callable(current):
            current = current()
    except TypeError:
        pass
    try:
        return float(current)
    except (TypeError, ValueError):
        return current


def load_metrics(path: Path, fallback: dict) -> dict:
    if path.exists():
        try:
            existing = json.loads(path.read_text(encoding="utf-8"))
            merged = fallback.copy()
            merged.update(existing)
            return merged
        except json.JSONDecodeError:
            print(f"WARNING: Could not parse existing metrics file: {path}")
    return fallback


def save_metrics(metrics: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"Saved metrics: {path}")


metrics_payload = metrics_template = {
    "model_id": MODEL_ID,
    "base_model": BASE_MODEL,
    "role": ROLE,
    "dataset_yaml": str(DATA_YAML) if DATA_YAML else "TBD",
    "status": "trained_or_validated" if val_metrics is not None else ("training_failed" if train_error else ("training_attempted" if train_result is not None else "not_run")),
    "errors": {
        "model_load_error": model_load_error,
        "train_error": train_error,
        "val_error": val_error,
    },
    "train_config": {
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "device": DEVICE,
    },
    "dataset_metrics": {
        "mAP50": metric_value(val_metrics, "box.map50"),
        "mAP50_95": metric_value(val_metrics, "box.map"),
        "precision": metric_value(val_metrics, "box.mp"),
        "recall": metric_value(val_metrics, "box.mr"),
    },
    "runtime_metrics": {
        "avg_inference_ms_per_image": "TBD",
        "fps_estimate": "TBD",
    },
    "qualitative_outputs": {
        "sample_prediction_dir": "experiments/yolo11s_ppe/runs/predict_samples",
        "video_prediction_dir": "TBD",
    },
    "risk_signal_readiness": {
        "person_detection_supported": True,
        "helmet_detection_supported": True,
        "no_helmet_or_head_supported": True,
        "danger_zone_supported_by_postprocessing": True,
        "notes": "Raw YOLO boxes/classes/confidences can feed person/helmet/head signals; danger zones are post-processing.",
    },
    "selection_notes": "Smoke experiment only. Final selection requires shared metrics, speed, qualitative errors, and risk-signal readiness.",
}
save_metrics(metrics_payload, METRICS_PATH)

## Shared Evaluation Sample Selection

Use the exact same fixed sample images across YOLOv11s, YOLOv11s, and the public PPE baseline. This keeps qualitative comparison and runtime benchmarking consistent.

In [ ]:
import json
import random
from pathlib import Path

SAMPLE_COUNT = 12
RANDOM_SEED = 42
SHARED_SAMPLE_PATH = Path("experiments/shared_eval_samples.json")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _collect_images_from_dir(image_dir: Path | None) -> list[Path]:
    if image_dir is None or not image_dir.exists():
        return []
    return sorted(p.resolve() for p in image_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def _is_under(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except Exception:
        return False


def _load_shared_samples(required_root: Path) -> list[Path]:
    if not SHARED_SAMPLE_PATH.exists():
        return []
    try:
        payload = json.loads(SHARED_SAMPLE_PATH.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read shared sample file; will recreate it. Error: {exc}")
        return []

    raw_paths = payload.get("sample_images", []) if isinstance(payload, dict) else payload
    if not isinstance(raw_paths, list):
        return []

    resolved = []
    for raw in raw_paths:
        raw_path = Path(raw)
        candidates = []
        if raw_path.is_absolute():
            candidates.append(raw_path)
        else:
            candidates.append(PROJECT_ROOT / raw_path)
            candidates.append(raw_path)

        existing = next((candidate.resolve() for candidate in candidates if candidate.exists()), None)
        if existing is not None and _is_under(existing, required_root):
            resolved.append(existing)

    if len(resolved) >= SAMPLE_COUNT:
        print(f"Reusing {len(resolved)} processed shared evaluation samples from: {SHARED_SAMPLE_PATH}")
        return resolved[:SAMPLE_COUNT]

    if raw_paths:
        print("Existing shared sample file is not from processed dataset, so it will be recreated.")
    return []


def _save_shared_samples(paths: list[Path], source_note: str) -> None:
    SHARED_SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    relative_paths = []
    for path in paths:
        try:
            relative_paths.append(path.resolve().relative_to(PROJECT_ROOT).as_posix())
        except Exception:
            relative_paths.append(str(path))
    SHARED_SAMPLE_PATH.write_text(
        json.dumps(
            {
                "source": source_note,
                "sample_count": len(relative_paths),
                "sample_images": relative_paths,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved shared evaluation samples to: {SHARED_SAMPLE_PATH}")


def select_shared_sample_images() -> list[Path]:
    required_root = dataset_root.resolve()
    existing_samples = _load_shared_samples(required_root)
    if existing_samples:
        return existing_samples

    candidate_dirs = [
        test_img_dir,
        val_img_dir,
        train_img_dir,
    ]

    for candidate_dir in candidate_dirs:
        images = _collect_images_from_dir(candidate_dir)
        if images:
            random.seed(RANDOM_SEED)
            selected = images if len(images) <= SAMPLE_COUNT else random.sample(images, SAMPLE_COUNT)
            selected = sorted(selected)
            source_note = str(candidate_dir)
            print(f"Selected {len(selected)} samples from processed dataset: {source_note}")
            _save_shared_samples(selected, source_note)
            return selected

    raise RuntimeError("No processed sample images found. Check data/processed/ppe_yolo/images/{train,val,test}.")


sample_images = select_shared_sample_images()
print(f"Sample count: {len(sample_images)}")
for path in sample_images[:12]:
    try:
        print(" -", path.relative_to(PROJECT_ROOT))
    except Exception:
        print(" -", path)


## Sample Prediction, Clean FPS Benchmark, and Visual Review

This section separates **qualitative visualization** from **runtime benchmarking**:

- `save=True` prediction is used only to create annotated images for visual inspection.
- Clean FPS is measured with `save=False`, after a warmup run, so it is not dominated by drawing/saving files.
- The resulting `runtime_metrics` is the value that should be used for model comparison.

In [ ]:
from collections import defaultdict
from pathlib import Path
import shutil
import time

try:
    import pandas as pd
except Exception:
    pd = None

sample_prediction_dir = None
detection_records = []
visual_results = []
clean_benchmark_results = []

runtime_metrics = {
    "avg_inference_ms_per_image": "TBD",
    "fps_estimate": "TBD",
    "benchmark_type": "TBD",
}

# Guard against running this cell before the shared sample selection cell.
sample_images = globals().get("sample_images", [])


def _class_name_from_id(class_id: int) -> str:
    names = getattr(model, "names", {}) if model is not None else {}
    if isinstance(names, dict):
        return str(names.get(class_id, class_id))
    if isinstance(names, list) and 0 <= class_id < len(names):
        return str(names[class_id])
    return str(class_id)


def _extract_detection_records(results) -> list[dict]:
    records = []
    for result in results or []:
        source_path = str(getattr(result, "path", ""))
        boxes = getattr(result, "boxes", None)
        if boxes is None:
            continue
        for box in boxes:
            try:
                class_id = int(box.cls.item())
                confidence = float(box.conf.item())
            except Exception:
                continue
            records.append(
                {
                    "source": source_path,
                    "image": Path(source_path).name if source_path else "TBD",
                    "class_id": class_id,
                    "class_name": _class_name_from_id(class_id),
                    "confidence": confidence,
                }
            )
    return records


def _average_result_speed(results) -> dict:
    """Average Ultralytics per-image preprocess/inference/postprocess speed if available."""
    speed_buckets = defaultdict(list)
    for result in results or []:
        speed = getattr(result, "speed", None)
        if not isinstance(speed, dict):
            continue
        for key, value in speed.items():
            try:
                speed_buckets[key].append(float(value))
            except Exception:
                pass
    return {
        f"avg_{key}_ms_per_image": sum(values) / len(values)
        for key, values in speed_buckets.items()
        if values
    }


if model is None:
    print("Skipping YOLOv11s sample prediction because model is not loaded.")
elif not sample_images:
    print("No sample images available; skipping YOLOv11s sample prediction.")
else:
    PREDICT_PROJECT = (PROJECT_ROOT / "experiments/yolo11s_ppe/runs").resolve()
    sample_prediction_dir = PREDICT_PROJECT / "predict_samples"
    PREDICT_PROJECT.mkdir(parents=True, exist_ok=True)

    # Keep the visual output folder deterministic and clean.
    if sample_prediction_dir.exists():
        shutil.rmtree(sample_prediction_dir)

    print("Saving annotated qualitative YOLOv11s predictions...")
    visual_results = model.predict(
        source=[str(path) for path in sample_images],
        conf=0.4,
        save=True,
        project=str(PREDICT_PROJECT),
        name="predict_samples",
        exist_ok=True,
        device=DEVICE,
    )
    detection_records = _extract_detection_records(visual_results)

    # Warmup is important; the first GPU call includes setup overhead.
    warmup_images = sample_images[: min(2, len(sample_images))]
    print("Running warmup prediction with save=False...")
    _ = model.predict(
        source=[str(path) for path in warmup_images],
        conf=0.4,
        save=False,
        verbose=False,
        device=DEVICE,
    )

    # Clean runtime benchmark: no drawing, no saving, repeated for more stable average.
    BENCHMARK_REPEATS = 3
    print(f"Running clean FPS benchmark: save=False, repeats={BENCHMARK_REPEATS}...")
    elapsed_values = []
    total_images = 0
    clean_benchmark_results = []

    for repeat_idx in range(BENCHMARK_REPEATS):
        start = time.perf_counter()
        batch_results = model.predict(
            source=[str(path) for path in sample_images],
            conf=0.4,
            save=False,
            verbose=False,
            device=DEVICE,
        )
        elapsed = time.perf_counter() - start
        elapsed_values.append(elapsed)
        total_images += len(sample_images)
        clean_benchmark_results.extend(batch_results)
        print(f"Repeat {repeat_idx + 1}/{BENCHMARK_REPEATS}: {elapsed:.3f}s total")

    total_elapsed = sum(elapsed_values)
    avg_ms = (total_elapsed / total_images) * 1000 if total_images else None
    fps = (1000 / avg_ms) if avg_ms and avg_ms > 0 else None
    ultralytics_speed = _average_result_speed(clean_benchmark_results)

    runtime_metrics = {
        "avg_inference_ms_per_image": avg_ms,
        "fps_estimate": fps,
        "benchmark_type": "clean_predict_no_save_after_warmup",
        "sample_count": len(sample_images),
        "benchmark_repeats": BENCHMARK_REPEATS,
        "total_images_timed": total_images,
        "total_elapsed_seconds": total_elapsed,
        "elapsed_seconds_per_repeat": elapsed_values,
        "ultralytics_speed": ultralytics_speed,
        "note": "Clean FPS excludes drawing, image saving, and qualitative visualization overhead.",
    }

    print("\nClean runtime metrics")
    print(json.dumps(runtime_metrics, indent=2))
    print(f"Annotated predictions saved to: {sample_prediction_dir}")

    metrics_payload = load_metrics(METRICS_PATH, metrics_payload)
    metrics_payload["runtime_metrics"] = runtime_metrics
    metrics_payload.setdefault("qualitative_outputs", {})["sample_prediction_dir"] = str(sample_prediction_dir)
    metrics_payload["status"] = "sample_prediction_completed"
    save_metrics(metrics_payload, METRICS_PATH)


## Visualize Selected Evaluation Samples

Use this cell to quickly confirm which images are used for the shared qualitative comparison.

In [ ]:
from PIL import Image
import math
import matplotlib.pyplot as plt


def show_image_grid(paths, title="Image grid", max_images=12, cols=3, figsize_per_image=(4, 3)):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print(f"No images found for: {title}")
        return

    paths = paths[:max_images]
    rows = math.ceil(len(paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_image[0], rows * figsize_per_image[1]))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1:
        axes = list(axes)
    else:
        axes = list(axes.flatten())

    for ax, path in zip(axes, paths):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(path.name[:45], fontsize=9)
        ax.axis("off")

    for ax in axes[len(paths):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


sample_images = globals().get("sample_images", [])
show_image_grid(sample_images, title="Shared evaluation samples", max_images=12, cols=3)


## Visualize Training / Validation Artifacts

YOLO training usually saves useful charts such as `results.png`, `confusion_matrix.png`, PR curves, and label distribution images. These are helpful for a quick visual sanity check after training.

In [ ]:
TRAIN_ARTIFACT_DIRS = [
    PROJECT_ROOT / "experiments/yolo11s_ppe/runs/smoke_train",
    PROJECT_ROOT / "experiments/yolo11s_ppe/runs/val",
    # Fallback for older cells / Ultralytics path nesting if relative project paths were used.
    PROJECT_ROOT / "runs/detect/experiments/yolo11s_ppe/runs/smoke_train",
    PROJECT_ROOT / "runs/detect/experiments/yolo11s_ppe/runs/val",
]
ARTIFACT_NAMES = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "labels.jpg",
    "labels_correlogram.jpg",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

artifact_paths = []
seen_artifacts = set()
for artifact_dir in TRAIN_ARTIFACT_DIRS:
    if not artifact_dir.exists():
        continue
    for name in ARTIFACT_NAMES:
        candidate = artifact_dir / name
        if candidate.exists() and candidate.resolve() not in seen_artifacts:
            artifact_paths.append(candidate)
            seen_artifacts.add(candidate.resolve())

# Extra fallback: search inside experiment folder for common artifact names.
for candidate in (PROJECT_ROOT / "experiments/yolo11s_ppe").rglob("*.png"):
    if candidate.name in ARTIFACT_NAMES and candidate.resolve() not in seen_artifacts:
        artifact_paths.append(candidate)
        seen_artifacts.add(candidate.resolve())

print(f"Training/validation artifact images found: {len(artifact_paths)}")
for path in artifact_paths:
    try:
        print(" -", path.relative_to(PROJECT_ROOT))
    except Exception:
        print(" -", path)

show_image_grid(
    artifact_paths,
    title="YOLOv11s training / validation artifacts",
    max_images=9,
    cols=3,
    figsize_per_image=(5, 4),
)


## Visualize Annotated Predictions

These annotated images are the qualitative output from YOLOv11s. Use this to check whether the model detects `person`, `helmet`, and `head/no-helmet` in a way that is useful for future risk signals.

In [ ]:
prediction_image_paths = []
if sample_prediction_dir is not None and Path(sample_prediction_dir).exists():
    prediction_image_paths = sorted(
        path for path in Path(sample_prediction_dir).rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )
else:
    fallback_predict_dir = PROJECT_ROOT / "experiments/yolo11s_ppe/runs/predict_samples"
    if fallback_predict_dir.exists():
        sample_prediction_dir = fallback_predict_dir
        prediction_image_paths = sorted(
            path for path in fallback_predict_dir.rglob("*")
            if path.suffix.lower() in IMAGE_EXTENSIONS
        )

print(f"Annotated prediction images found: {len(prediction_image_paths)}")
if sample_prediction_dir is not None:
    print("Prediction directory:", sample_prediction_dir)

show_image_grid(prediction_image_paths, title="Annotated YOLOv11s predictions", max_images=12, cols=3)


## Detection Summary Visualization

This is not mAP. It is a visual/qualitative summary of what YOLOv11s detected on the shared sample images.

In [ ]:
if not detection_records:
    print("No detection records available. Run the sample prediction cell first.")
else:
    if pd is not None:
        detections_df = pd.DataFrame(detection_records)
        display(detections_df.head(30))
    else:
        detections_df = None
        print(detection_records[:10])

    class_counts = {}
    confidence_values = []
    for record in detection_records:
        class_counts[record["class_name"]] = class_counts.get(record["class_name"], 0) + 1
        confidence_values.append(record["confidence"])

    plt.figure(figsize=(10, 4))
    names = list(class_counts.keys())
    counts = [class_counts[name] for name in names]
    plt.bar(names, counts)
    plt.title("Detection count by YOLOv11s class")
    plt.ylabel("Box count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(confidence_values, bins=10)
    plt.title("Detection confidence distribution")
    plt.xlabel("Confidence")
    plt.ylabel("Box count")
    plt.tight_layout()
    plt.show()

    if pd is not None:
        summary_path = EXPERIMENT_DIR / "sample_detection_summary.csv"
        detections_df.to_csv(summary_path, index=False)
        print(f"Saved detection summary: {summary_path}")


## Risk-Signal Readiness Preview

This preview maps YOLOv11s detections into the MVP risk-signal vocabulary. It does **not** implement final risk scoring yet; it only shows whether each image contains signals that could later feed the Rule Engine.

Note: the Roboflow fallback label order may be `0=head, 1=helmet, 2=person`, while the final project schema is `0=person, 1=helmet, 2=head`. This smoke notebook uses the active model/data class names and normalizes them for risk-signal preview.

In [ ]:
def normalize_project_class(name: str) -> str:
    value = str(name).strip().lower().replace("-", "_").replace(" ", "_")
    if value in {"person", "worker", "people"}:
        return "person"
    if value in {"helmet", "hardhat", "hard_hat", "safety_helmet", "hard_hat_workers"}:
        return "helmet"
    if value in {"head", "no_helmet", "no_hardhat", "no_hard_hat", "bare_head", "no_hardhat"}:
        return "head"
    return value


if not detection_records:
    print("No detection records available. Run the sample prediction cell first.")
else:
    grouped = defaultdict(list)
    for record in detection_records:
        grouped[record["source"]].append(record)

    risk_signal_rows = []
    for source, records in grouped.items():
        normalized_records = [(normalize_project_class(r["class_name"]), r) for r in records]
        normalized_names = [item[0] for item in normalized_records]
        person_confidences = [r["confidence"] for name, r in normalized_records if name == "person"]
        helmet_confidences = [r["confidence"] for name, r in normalized_records if name == "helmet"]
        head_confidences = [r["confidence"] for name, r in normalized_records if name == "head"]

        risk_signal_rows.append(
            {
                "image": Path(source).name,
                "person_detected": "person" in normalized_names,
                "helmet_detected": "helmet" in normalized_names,
                "head_or_no_helmet_detected": "head" in normalized_names,
                "person_confidence_max": max(person_confidences) if person_confidences else None,
                "helmet_confidence_max": max(helmet_confidences) if helmet_confidences else None,
                "no_helmet_confidence_max": max(head_confidences) if head_confidences else None,
                "box_count": len(records),
                "risk_ready_note": "inside_danger_zone and stay_time_seconds require video ROI/tracking in later phases.",
            }
        )

    if pd is not None:
        risk_signal_df = pd.DataFrame(risk_signal_rows).sort_values("image")
        display(risk_signal_df)
        preview_path = EXPERIMENT_DIR / "risk_signal_preview.csv"
        risk_signal_df.to_csv(preview_path, index=False)
        print(f"Saved risk-signal preview: {preview_path}")
    else:
        for row in risk_signal_rows:
            print(row)


## Optional Video Prediction

Video inference is used for cross-verification and demo realism. It is **not** the primary quantitative benchmark for model selection.

In [ ]:
VIDEO_SOURCES = {
    "camera_1": Path("video_source/camera_1/cam1.mp4"),
    "camera_2": Path("video_source/camera_2/cam2.mp4"),
    "camera_3": Path("video_source/camera_3/cam3.mp4"),
}

video_output_dirs = {}
if model is None:
    print("Skipping video prediction because model is not loaded.")
else:
    VIDEO_PREDICT_PROJECT = (PROJECT_ROOT / "experiments/yolo11s_ppe/runs").resolve()
    VIDEO_PREDICT_PROJECT.mkdir(parents=True, exist_ok=True)

    for camera_id, video_path in VIDEO_SOURCES.items():
        if not video_path.exists():
            print(f"Skipping {camera_id}; video not found: {video_path}")
            continue

        output_name = f"predict_video_{camera_id}"
        output_dir = VIDEO_PREDICT_PROJECT / output_name
        if output_dir.exists():
            shutil.rmtree(output_dir)

        print(f"Running video prediction for {camera_id}: {video_path}")
        model.predict(
            source=str(video_path),
            conf=0.4,
            save=True,
            project=str(VIDEO_PREDICT_PROJECT),
            name=output_name,
            exist_ok=True,
            device=DEVICE,
        )
        video_output_dirs[camera_id] = str(output_dir)
        print(f"Saved {camera_id} video prediction to: {output_dir}")

    metrics_payload = load_metrics(METRICS_PATH, metrics_payload)
    metrics_payload.setdefault("qualitative_outputs", {})["video_prediction_dir"] = video_output_dirs or "TBD"
    save_metrics(metrics_payload, METRICS_PATH)


## Visualize Video Prediction Outputs

If camera videos exist and video prediction was run, this cell previews the generated output video. AVI outputs are converted to MP4 for easier Colab display.

In [ ]:
from IPython.display import HTML, display


def _find_video_files(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(
        path for path in folder.rglob("*")
        if path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}
    )


for camera_id, output_dir_text in video_output_dirs.items():
    output_dir = Path(output_dir_text)
    videos = _find_video_files(output_dir)
    print(f"{camera_id}: {len(videos)} video file(s) found in {output_dir}")
    if not videos:
        continue

    video_path = videos[0]
    preview_path = video_path

    if video_path.suffix.lower() != ".mp4":
        preview_path = video_path.with_suffix(".mp4")
        if not preview_path.exists():
            print(f"Converting {video_path.name} to MP4 preview...")
            !ffmpeg -y -i "{video_path}" -vcodec libx264 -acodec aac "{preview_path}" >/dev/null 2>&1

    print(f"Previewing {camera_id}: {preview_path}")
    html = f"""
    <video width="800" controls>
      <source src="{preview_path.as_posix()}" type="video/mp4">
      Your browser does not support the video tag.
    </video>
    """
    display(HTML(html))


## Metrics JSON Preview

Quickly inspect the final standardized `metrics.json` after training, validation, sample prediction, runtime benchmark, and optional video prediction.

In [ ]:
if METRICS_PATH.exists():
    metrics_payload = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print(json.dumps(metrics_payload, indent=2)[:5000])
else:
    print(f"Metrics file not found: {METRICS_PATH}")


## Risk Signal Bridge Note

YOLO output boxes/classes/confidences are raw perception outputs. Later phases should convert them into:

- `person_detected`
- `helmet_detected`
- `head_detected`
- `no_helmet_confidence`
- `inside_danger_zone`
- `stay_time_seconds`
- `moving_toward_zone`

This notebook does not implement final Risk Scoring yet.

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/factory-safety-ai-cctv"

!mkdir -p "$DRIVE_PROJECT/experiments"
!rm -rf "$DRIVE_PROJECT/experiments/yolo11s_ppe"
!cp -r experiments/yolo11s_ppe "$DRIVE_PROJECT/experiments/yolo11s_ppe"

!cp experiments/shared_eval_samples.json "$DRIVE_PROJECT/experiments/shared_eval_samples.json" 2>/dev/null || true

print("Synced yolo11s_ppe results back to Google Drive.")
